# work6 v6 换词版（2026-08-13）：命令词"关闭"→"关"（双不送气塞音线索被板端底噪掩蔽，改押 uan 低频长韵尾）。
# 改动：WANTED 换词（类别序号不变，板端零改动）；新增剪字 cell；恢复从头训练（lr=1e-3, epochs=100）；归一化仍冻结 v4。
# 上传清单：v4 三件套(h5+npz+tflite) + 全部原始 m4a + noise_*/recite_* 素材 + 板端关闭_入训1.wav + 板端底噪_16k.wav + 板端关闭_考题_勿入训.wav

cell0:归一化检测

In [3]:
import numpy as np
p = np.load('/content/kimi_kws_0724_4cls_v4_norm_params.npz')
print(list(p.keys()))
m = p['mean'] if 'mean' in p else p[list(p.keys())[0]]
print(m.reshape(-1)[:3])
# 应打印约 [-15.944, -14.166, -12.998]，与板端代码 MEAN_F 前三个一致 → 冻结成立

['mean', 'std']
[[[-15.943873  -14.16633   -12.998032  -12.462342  -12.318329
   -12.704891  -12.857549  -12.660684  -12.195224  -12.259253
   -12.342042  -12.198825  -12.3698    -12.4610615 -12.394521
   -12.323932  -12.414016  -12.5005245 -12.309226  -12.138959
   -12.1677    -12.141489  -11.859203  -11.67147   -11.778436
   -11.855022  -11.824253  -11.767101  -11.634663  -11.674359
   -11.811094  -11.844134  -11.938671  -12.157752  -12.199463
   -12.262533  -12.318928  -12.540266  -13.091838  -14.084435 ]]]


Cell 1：环境 + 配置 + 发现录音文件

In [ ]:
!pip install pydub librosa edge-tts -q

import os, math, glob, time, random, shutil, subprocess
import numpy as np
import librosa
import tensorflow as tf
from sklearn.model_selection import train_test_split
from pydub import AudioSegment
from tensorflow.keras import layers, models
from datetime import date

SAMPLE_RATE = 16000
FRAME_LEN = 480; FRAME_STEP = 320; NFFT = 512
NUM_MEL_BINS = 40; NUM_FRAMES = 49
WANTED = ["打开","关","唤醒"]   # v6: 关闭→关，类别序号不变
NUM_CLASSES = len(WANTED) + 1
w2i = {w:i+1 for i,w in enumerate(WANTED)}
w2i["negative"]=0
i2w = {v:k for k,v in w2i.items()}
DATA_DIR = "/content/my_dataset"
DATE_STR = date.today().strftime("%m%d")
TEST_PEOPLE = 1
random.seed(42); np.random.seed(42)

for w in WANTED:
    os.makedirs(f"{DATA_DIR}/{w}", exist_ok=True)
print(f"words: {WANTED}  classes: {NUM_CLASSES}  date: {DATE_STR}")

print("=== discovering files ===")
PERSON_FILES = []
for word in WANTED:
    for f in glob.glob(f"/content/{word}_*.m4a") + glob.glob(f"/content/{word}_*.mp3"):
        basename = os.path.basename(f)
        tag = basename.replace(f"{word}_", "").replace(".m4a", "").replace(".mp3", "")
        PERSON_FILES.append((word, tag, f))
PERSON_TAGS = sorted(set(tag for _, tag, _ in PERSON_FILES))
print(f"{len(PERSON_TAGS)} people: {PERSON_TAGS}")
EXCLUDE_TAGS = {"男6"}
PERSON_FILES = [(w, t, f) for (w, t, f) in PERSON_FILES if t not in EXCLUDE_TAGS]
PERSON_TAGS = sorted(set(tag for _, tag, _ in PERSON_FILES))
print(f"after exclusion: {len(PERSON_TAGS)} people: {PERSON_TAGS}")
assert len(PERSON_TAGS) > TEST_PEOPLE, "need at least TEST_PEOPLE+1 people"

words: ['打开', '关闭', '唤醒']  classes: 4  date: 0724
=== discovering files ===
10 people: ['女1', '女2', '女3', '女4', '男1', '男2', '男3', '男4', '男5', '男6']
after exclusion: 9 people: ['女1', '女2', '女3', '女4', '男1', '男2', '男3', '男4', '男5']


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Cell 2：切分录音 → wav 片段

In [ ]:
for w in WANTED:
    for f in glob.glob(f"{DATA_DIR}/{w}/{w}_男6_*.wav"):
        os.remove(f)
def robust_split(audio_seg):
    samples = np.array(audio_seg.get_array_of_samples()).astype(np.float32)
    rms = librosa.feature.rms(y=samples, frame_length=256, hop_length=128)[0]
    threshold = np.median(rms[rms > 0]) * 1.8
    if threshold < 0.001: threshold = 0.01
    is_voice = rms > threshold
    padded = np.concatenate([[False], is_voice, [False]])
    starts = np.where(~padded[:-1] & padded[1:])[0]
    ends = np.where(padded[:-1] & ~padded[1:])[0]
    pad_frames = 10
    segments = []
    for s, e in zip(starts, ends):
        s = max(0, s - pad_frames); e = min(len(is_voice), e + pad_frames)
        start_sample = s * 128; end_sample = min(e * 128, len(samples))
        dur_ms = (end_sample - start_sample) / SAMPLE_RATE * 1000
        segments.append((start_sample, end_sample, dur_ms))

    for merge_gap in [200, 300, 400, 600, 800]:
        segs = segments.copy()
        changed = True
        while changed:
            changed = False
            i = 0
            while i < len(segs) - 1:
                gap_ms = (segs[i+1][0] - segs[i][1]) / SAMPLE_RATE * 1000
                if gap_ms < merge_gap:
                    segs[i] = (segs[i][0], segs[i+1][1], segs[i][2] + segs[i+1][2] + gap_ms)
                    segs.pop(i+1)
                    changed = True
                else:
                    i += 1
        valid = [s for s in segs if 200 < s[2] < 2000]
        med = np.median([v[2] for v in valid]) if valid else 0
        if (8 <= len(valid) <= 60 and med >= 450) or merge_gap == 800:
            segments = segs
            print(f"      merge_gap={merge_gap}ms, valid={len(valid)}, median={med/1000:.2f}s")
            break


    result = []
    for start, end, dur_ms in segments:
        if 200 < dur_ms < 2000:
            seg_samples = samples[start:end]
            seg = AudioSegment(seg_samples.astype(np.int16).tobytes(),
                               frame_rate=SAMPLE_RATE, sample_width=2, channels=1)
            result.append(seg)
    return result[:40]

for word, tag, path in PERSON_FILES:
    for old_f in glob.glob(f"{DATA_DIR}/{word}/{word}_{tag}_*.wav"): os.remove(old_f)
    audio = AudioSegment.from_file(path).set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
    valid = robust_split(audio)
    for i, chunk in enumerate(valid):
        chunk.export(f"{DATA_DIR}/{word}/{word}_{tag}_{i:03d}.wav", format="wav")
    print(f"  {word}_{tag}: {len(valid)} clips")

for w in WANTED:
    for f in glob.glob(f"{DATA_DIR}/{w}/*.wav"):
        if os.path.getsize(f) < 100: os.remove(f)
print("\nclip counts:")
for w in WANTED:
    print(f"  {w}: {len(glob.glob(f'{DATA_DIR}/{w}/*.wav'))}")

      merge_gap=200ms, valid=31, median=0.71s
  打开_男2: 31 clips
      merge_gap=200ms, valid=34, median=0.72s
  打开_男4: 34 clips
      merge_gap=200ms, valid=30, median=0.77s
  打开_男5: 30 clips
      merge_gap=200ms, valid=31, median=0.79s
  打开_女4: 31 clips
      merge_gap=200ms, valid=12, median=0.90s
  打开_女1: 12 clips
      merge_gap=200ms, valid=25, median=0.75s
  打开_男1: 25 clips
      merge_gap=200ms, valid=26, median=0.81s
  打开_男3: 26 clips
      merge_gap=200ms, valid=30, median=0.76s
  打开_女2: 30 clips
      merge_gap=200ms, valid=23, median=0.64s
  打开_女3: 23 clips
      merge_gap=200ms, valid=31, median=0.98s
  关闭_男3: 31 clips
      merge_gap=200ms, valid=25, median=0.70s
  关闭_男2: 25 clips
      merge_gap=200ms, valid=29, median=0.76s
  关闭_女2: 29 clips
      merge_gap=200ms, valid=30, median=0.66s
  关闭_女4: 30 clips
      merge_gap=200ms, valid=25, median=0.77s
  关闭_男1: 25 clips
      merge_gap=200ms, valid=17, median=0.96s
  关闭_女1: 17 clips
      merge_gap=200ms, valid=27, median=

Cell 3：特征提取函数 + 通用小工具

In [ ]:
def extract_log_mel(audio):
    audio = np.append(audio[0], audio[1:]-0.97*audio[:-1])
    nf = 1+(len(audio)-FRAME_LEN)//FRAME_STEP
    if nf>NUM_FRAMES: nf=NUM_FRAMES
    frames = np.zeros((nf,FRAME_LEN))
    win = np.hamming(FRAME_LEN)
    for i in range(nf):
        frames[i] = audio[i*FRAME_STEP:i*FRAME_STEP+FRAME_LEN]*win
    ps = (np.abs(np.fft.rfft(frames,n=NFFT))**2)/NFFT
    low,high = 0, 2595*np.log10(1+SAMPLE_RATE/1400)
    pts = np.linspace(low,high,NUM_MEL_BINS+2)
    hz = 700*(10**(pts/2595)-1)
    bins = np.floor((NFFT+1)*hz/SAMPLE_RATE).astype(int); bins = np.clip(bins,0,NFFT//2)
    fb = np.zeros((NUM_MEL_BINS,NFFT//2+1))
    for m in range(1,NUM_MEL_BINS+1):
        for k in range(bins[m-1],bins[m]): fb[m-1,k]=(k-bins[m-1])/max(bins[m]-bins[m-1],1)
        for k in range(bins[m],bins[m+1]): fb[m-1,k]=(bins[m+1]-k)/max(bins[m+1]-bins[m],1)
    return np.log(np.where((e:=np.dot(ps,fb.T))==0,np.finfo(float).eps,e)).astype(np.float32)

def to_1s(a):
    a = a.astype(np.float32)
    if len(a) < SAMPLE_RATE: a = np.pad(a,(0,SAMPLE_RATE-len(a)))
    else: a = a[:SAMPLE_RATE]
    return a

def load_wav(f):
    raw = tf.io.read_file(f)
    a, _ = tf.audio.decode_wav(raw, desired_channels=1)
    return to_1s(tf.squeeze(a,-1).numpy().astype(np.float32))

print("feature extractor ready")

feature extractor ready


Cell 4：环境底噪 → 噪声库

In [ ]:
noise_pool = []
nf = sorted(glob.glob("/content/noise_*.m4a") + glob.glob("/content/noise_*.mp3") + glob.glob("/content/noise_*.wav"))
for f in nf:
    a = AudioSegment.from_file(f).set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
    s = np.array(a.get_array_of_samples()).astype(np.float32) / 32768.0
    for st in range(0, max(1, len(s)-SAMPLE_RATE), SAMPLE_RATE//2):
        seg = s[st:st+SAMPLE_RATE]
        if len(seg) == SAMPLE_RATE and np.sqrt(np.mean(seg**2)) > 1e-4:
            noise_pool.append(seg.astype(np.float32))

if len(noise_pool) == 0:
    print("no noise_* file found, synthetic colored noise fallback")
    for i in range(80):
        w = np.random.randn(SAMPLE_RATE).astype(np.float32)
        w = np.convolve(w, np.ones(32)/32, mode="same")
        w = w/(np.max(np.abs(w))+1e-9)*random.uniform(0.01,0.06)
        noise_pool.append(w.astype(np.float32))

def mix_with_noise(a, snr_db=None):
    if snr_db is None: snr_db = random.uniform(5, 20)
    n = noise_pool[random.randrange(len(noise_pool))]
    if len(n) < len(a): n = np.tile(n, math.ceil(len(a)/len(n)))[:len(a)]
    else:
        st = random.randint(0, len(n)-len(a)); n = n[st:st+len(a)]
    a_rms = np.sqrt(np.mean(a**2)+1e-10); n_rms = np.sqrt(np.mean(n**2)+1e-10)
    return (a + n*(a_rms/(10**(snr_db/20)))/(n_rms+1e-10)).astype(np.float32)

print(f"noise clips in pool: {len(noise_pool)}")

noise clips in pool: 198


Cell 4.5（新增 v5）：板端真实底噪入库（占噪声池 25%，勿超）

In [ ]:
# 把板端底噪切 1s 段入 noise_pool；条数 = 原池 1/3 → 被抽中概率 = 25%
import numpy as np
ba = AudioSegment.from_file("/content/板端底噪_16k.wav").set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
bs = np.array(ba.get_array_of_samples()).astype(np.float32) / 32768.0
board_noise = []
for st in range(0, len(bs)-SAMPLE_RATE+1, SAMPLE_RATE//2):
    seg = bs[st:st+SAMPLE_RATE]
    if len(seg) == SAMPLE_RATE and np.sqrt(np.mean(seg**2)) > 1e-4:
        board_noise.append(seg.astype(np.float32))
n_add = max(1, len(noise_pool)//3)
for i in range(n_add):
    noise_pool.append(board_noise[i % len(board_noise)])
print(f"板端底噪段 {len(board_noise)} 条，重复入库 {n_add} 条；噪声池 {len(noise_pool)}，板端占比 {n_add/len(noise_pool)*100:.1f}%")

Cell 4.6（v5r2 并入 v6）：板端信道仿真器 boardify

In [ ]:
# ===== 板端信道仿真器（变量名对齐本 notebook）=====
import scipy.signal as sg

bnoise, _ = librosa.load("/content/板端底噪_16k.wav", sr=SAMPLE_RATE)

_rms = []
for f in sorted(glob.glob(f"{DATA_DIR}/关闭/关闭_*_*.wav"))[:50]:
    _rms.append(np.sqrt(np.mean(load_wav(f)**2)))
TARGET_RMS = float(np.median(_rms)) if _rms else 0.05
print("AGC 目标 RMS ≈", round(TARGET_RMS, 4))

def boardify(x, snr_db=None, offset=None):
    """干净1s语音 → 板端世界：限带6.8k → 15.5k往返 → 随机埋进底噪窗 → AGC式归一"""
    if snr_db is None: snr_db = np.random.choice([0, 5, 10, 15])
    if offset is None: offset = np.random.randint(0, 5000)
    sos = sg.butter(6, 6800, 'lowpass', fs=SAMPLE_RATE, output='sos')
    x = sg.sosfilt(sos, x)
    x = sg.resample_poly(sg.resample_poly(x, 155, 160), 160, 155)
    i = np.random.randint(0, len(bnoise) - SAMPLE_RATE)
    win = bnoise[i:i+SAMPLE_RATE].copy()
    L = min(len(x), SAMPLE_RATE - offset)
    seg = x[:L]
    sp = np.sqrt(np.mean(seg**2)) + 1e-9
    nsr = np.sqrt(np.mean(win**2)) + 1e-9
    win[offset:offset+L] += seg * (nsr * 10**(snr_db/20)) / sp
    win = win * TARGET_RMS / (np.sqrt(np.mean(win**2)) + 1e-9)
    return np.clip(win, -1, 1).astype(np.float32)

Cell 5：多音色 TTS unknown（含发音相近难负样本）

In [ ]:
UNK_DIR = f"{DATA_DIR}/_unknown"
if os.path.exists(UNK_DIR): shutil.rmtree(UNK_DIR)
os.makedirs(UNK_DIR, exist_ok=True)

UNK_WORDS = ["苹果","你好","今天","天气","不错","谢谢","好的","是的","电脑","音乐",
             "上午","下午","吃饭","睡觉","工作","学习","手机","电话","电视","空调",
             "灯光","窗户","门锁","路上","跑步","喝水","看书","写字","画画","唱歌",
             "高兴","漂亮","聪明","努力","简单","复杂","热闹","安静","干净","整齐",
             "春天","夏天","秋天","冬天","早上","晚上","左边","右边","前面","后面"]
HARD_NEG = ["打牌","打工","大哥","大街","大开","光笔","管壁","关窗","关灯",
            "换新","环形","患病","幻影","欢呼"]
ALL_UNK = UNK_WORDS + HARD_NEG

VOICES = ["zh-CN-XiaoxiaoNeural","zh-CN-YunxiNeural","zh-CN-XiaoyiNeural",
          "zh-CN-YunjianNeural","zh-CN-YunyangNeural","zh-CN-liaoning-XiaobeiNeural",
          "zh-CN-shaanxi-EmmaNeural"]

def tts_one(text, voice, out_mp3):
    if os.path.exists(out_mp3): os.remove(out_mp3)
    try:
        subprocess.run(["edge-tts","--voice",voice,"--text",text,"--write-media",out_mp3],
                       capture_output=True, timeout=30)
    except Exception:
        return False
    return os.path.exists(out_mp3) and os.path.getsize(out_mp3) > 1000

fail = 0
for word in ALL_UNK:
    for vi, voice in enumerate(VOICES):
        if not tts_one(word, voice, "/tmp/_u.mp3"):
            fail += 1; continue
        a = AudioSegment.from_file("/tmp/_u.mp3").set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
        s = np.array(a.get_array_of_samples()).astype(np.float32) / 32768.0
        base = f"{UNK_DIR}/unk_{word}_{vi}"
        v0 = to_1s(mix_with_noise(s, random.uniform(18, 28)))
        AudioSegment((np.clip(v0,-1,1)*32767).astype(np.int16).tobytes(),
                     frame_rate=SAMPLE_RATE, sample_width=2, channels=1).export(base+"_v0.wav", format="wav")
        for k in (1, 2):
            sp = librosa.resample(s, orig_sr=SAMPLE_RATE, target_sr=int(SAMPLE_RATE*random.choice([0.9,1.1])))
            aug = to_1s(mix_with_noise(sp, random.uniform(5, 18)) * random.uniform(0.5, 1.3))
            AudioSegment((np.clip(aug,-1,1)*32767).astype(np.int16).tobytes(),
                         frame_rate=SAMPLE_RATE, sample_width=2, channels=1).export(base+f"_v{k}.wav", format="wav")
print(f"unknown: {len(glob.glob(f'{UNK_DIR}/*.wav'))} files, tts failures: {fail}")

unknown: 1152 files, tts failures: 64


cell5.5真人unknown

In [ ]:
RECITE_FILES = sorted(glob.glob("/content/recite_*.m4a") + glob.glob("/content/recite_*.mp3")
                      + glob.glob("/content/recite_*.wav"))
MAX_RECITE_CLIPS = 150

def speech_regions(samples):
    rms = librosa.feature.rms(y=samples, frame_length=256, hop_length=128)[0]
    thr = np.median(rms[rms > 0]) * 1.5
    if thr < 0.001: thr = 0.01
    mask = rms > thr
    padded = np.concatenate([[False], mask, [False]])
    starts = np.where(~padded[:-1] & padded[1:])[0]
    ends = np.where(padded[:-1] & ~padded[1:])[0]
    regions = [[s*128, e*128] for s, e in zip(starts, ends)]
    merged = []
    for r in regions:
        if merged and r[0] - merged[-1][1] < int(0.3*SAMPLE_RATE):
            merged[-1][1] = r[1]
        else:
            merged.append(r)
    return merged

recite_count = 0
for f in RECITE_FILES:
    a = AudioSegment.from_file(f).set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
    s = np.array(a.get_array_of_samples()).astype(np.float32) / 32768.0
    for st, en in speech_regions(s):
        dur = (en - st) / SAMPLE_RATE
        if dur < 0.3: continue
        if dur <= 1.2:
            clips = [(st, en)]
        else:
            clips = [(p, p + SAMPLE_RATE) for p in range(st, en - SAMPLE_RATE, SAMPLE_RATE // 2)]
        for cs, ce in clips:
            if recite_count >= MAX_RECITE_CLIPS: break
            seg = s[cs:ce]
            AudioSegment((np.clip(seg,-1,1)*32767).astype(np.int16).tobytes(),
                         frame_rate=SAMPLE_RATE, sample_width=2, channels=1
                         ).export(f"{UNK_DIR}/unk_recite_{recite_count:04d}.wav", format="wav")
            recite_count += 1
print(f"recite unknown clips: {recite_count}")

recite unknown clips: 150


cell5.6英文短词负样本

In [ ]:
ENG_WORDS = ["yes","no","ok","hey","hello","stop","go","turn","light","open",
             "close","up","down","on","off","start","play","pause","next","back",
             "home","sure","right","what","when","please","thanks","sorry","maybe","always"]
ENG_VOICES = ["en-US-AriaNeural","en-US-GuyNeural","en-GB-SoniaNeural","en-AU-NatashaNeural"]

fail_eng = 0
for word in ENG_WORDS:
    for vi, voice in enumerate(ENG_VOICES):
        if not tts_one(word, voice, "/tmp/_e.mp3"):
            fail_eng += 1; continue
        a = AudioSegment.from_file("/tmp/_e.mp3").set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
        s = np.array(a.get_array_of_samples()).astype(np.float32) / 32768.0
        base = f"{UNK_DIR}/unk_eng_{word}_{vi}"
        v0 = to_1s(mix_with_noise(s, random.uniform(18, 28)))
        AudioSegment((np.clip(v0,-1,1)*32767).astype(np.int16).tobytes(),
                     frame_rate=SAMPLE_RATE, sample_width=2, channels=1).export(base+"_v0.wav", format="wav")
        for k in (1, 2):
            sp = librosa.resample(s, orig_sr=SAMPLE_RATE, target_sr=int(SAMPLE_RATE*random.choice([0.9,1.1])))
            aug = to_1s(mix_with_noise(sp, random.uniform(5, 18)) * random.uniform(0.5, 1.3))
            AudioSegment((np.clip(aug,-1,1)*32767).astype(np.int16).tobytes(),
                         frame_rate=SAMPLE_RATE, sample_width=2, channels=1).export(base+f"_v{k}.wav", format="wav")
print(f"english negatives done, total unknown files: {len(glob.glob(f'{UNK_DIR}/*.wav'))}, eng failures: {fail_eng}")

english negatives done, total unknown files: 1662, eng failures: 0


cell5.7

In [ ]:
STREAM_FILES = sorted(glob.glob("/content/recite_*.m4a") + glob.glob("/content/recite_*.mp3")
                      + glob.glob("/content/recite_*.wav"))
N_STREAM = 2500
stream_pool = []
for f in STREAM_FILES:
    a = AudioSegment.from_file(f).set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
    stream_pool.append(np.array(a.get_array_of_samples()).astype(np.float32) / 32768.0)
total_len = sum(len(s) for s in stream_pool)
print(f"stream source: {len(stream_pool)} files, {total_len/SAMPLE_RATE/60:.1f} min")

made = 0
while made < N_STREAM:
    s = stream_pool[random.randrange(len(stream_pool))]
    if len(s) <= SAMPLE_RATE: continue
    st = random.randint(0, len(s) - SAMPLE_RATE)
    seg = s[st:st + SAMPLE_RATE]
    if np.sqrt(np.mean(seg ** 2)) < 0.008: continue
    seg = seg * random.uniform(0.5, 1.3)
    if random.random() < 0.5: seg = mix_with_noise(seg, random.uniform(10, 25))
    seg = np.clip(seg, -1, 1)
    AudioSegment((seg * 32767).astype(np.int16).tobytes(),
                 frame_rate=SAMPLE_RATE, sample_width=2, channels=1
                 ).export(f"{UNK_DIR}/unk_strm_{made:04d}.wav", format="wav")
    made += 1
print(f"stream negatives added: {made}, total unknown: {len(glob.glob(f'{UNK_DIR}/*.wav'))}")

stream source: 1 files, 3.7 min
stream negatives added: 2500, total unknown: 4162


cell5.8修补数据补充

In [ ]:
# ========== Cell 5.8（v4）：mined 难负样本接入 + ×10 扩增 ==========
# 作用：把 WSL mine.py 挖出的作案音频（mine_*.wav）收编进负样本池
# 位置：Cell 5.7 之后、Cell 7 之前跑
# 输入：Colab 上传的 mine_*.wav（根目录散文件或 mined/ 文件夹均可）
# 输出：unk_mine_*.wav 写入与其他 unk_* 相同的目录，Cell 7 自动收编

import glob, os, random
import numpy as np
import librosa, soundfile as sf

random.seed(42); np.random.seed(42)
SR_ = globals().get("SR", 16000)

# ---------- 1. 找上传的 mined 文件（多路径兼容） ----------
mine_files = sorted(set(
    glob.glob("/content/mine_*.wav") +
    glob.glob("/content/mined/**/*.wav", recursive=True) +
    glob.glob("/content/**/mine_*.wav", recursive=True)
))
print(f"[5.8] 发现 mined 文件 {len(mine_files)} 条：")
for f in mine_files:
    print("    ", os.path.basename(f))
assert len(mine_files) > 0, "没找到 mine_*.wav！先把文件传进 Colab 再跑本 cell"

# ---------- 2. 找负样本目录（unk_recite/unk_strm 在哪就进哪） ----------
ref = (glob.glob("/content/**/unk_recite_*.wav", recursive=True) +
       glob.glob("/content/**/unk_strm_*.wav", recursive=True))
assert len(ref) > 0, "没找到 unk_recite/unk_strm，先跑 Cell 5.5 和 5.7 再来"
NEG_DIR = os.path.dirname(ref[0])
print(f"[5.8] 负样本目录：{NEG_DIR}")

# 幂等：清掉上一轮残留
for old in glob.glob(os.path.join(NEG_DIR, "unk_mine_*.wav")):
    os.remove(old)

# ---------- 3. 拷贝原件（统一 16k 单声道 1 秒，重命名 unk_mine_*） ----------
copied = []
for i, f in enumerate(mine_files):
    y, _ = librosa.load(f, sr=SR_, mono=True)
    y = np.asarray(y, dtype=np.float32)
    if len(y) > SR_:
        y = y[:SR_]
    elif len(y) < SR_:
        y = np.pad(y, (0, SR_ - len(y)))
    dst = os.path.join(NEG_DIR, f"unk_mine_{i:03d}_orig.wav")
    sf.write(dst, y, SR_)
    copied.append(y)
print(f"[5.8] 原件已拷贝 {len(copied)} 条")

# ---------- 4. ×10 扩增（配方与 Cell 7 一致：变速/变调/时移/增益/混噪） ----------
AUG_PER_CLIP = 10

def _rand_shift(y):
    return np.roll(y, random.randint(-int(0.2 * SR_), int(0.2 * SR_)))

def _rand_stretch(y):
    if random.random() < 0.5:
        rate = random.choice([0.85, 0.9, 1.1, 1.15])
        y2 = librosa.effects.time_stretch(y, rate=rate)
        if len(y2) > SR_:
            y2 = y2[:SR_]
        elif len(y2) < SR_:
            y2 = np.pad(y2, (0, SR_ - len(y2)))
        return y2
    return y

def _rand_pitch(y):
    return librosa.effects.pitch_shift(y, sr=SR_, n_steps=random.uniform(-3, 3))

def _rand_gain(y):
    return y * random.uniform(0.4, 1.4)

def _mix_noise(y):
    pool = globals().get("noise_pool", None)   # Cell 4 的真底噪池
    if not pool:
        return y
    n = pool[random.randrange(len(pool))]
    if isinstance(n, str):
        n, _ = librosa.load(n, sr=SR_, mono=True)
    if len(n) < len(y):
        n = np.tile(n, int(np.ceil(len(y) / len(n))))
    n = n[:len(y)]
    snr = random.uniform(5, 20)
    rms_y = np.sqrt(np.mean(y ** 2)) + 1e-8
    rms_n = np.sqrt(np.mean(n ** 2)) + 1e-8
    return y + n * (rms_y / (10 ** (snr / 20) * rms_n))

n_aug = 0
for i, y0 in enumerate(copied):
    for j in range(AUG_PER_CLIP):
        y = _rand_shift(y0.copy())
        y = _rand_stretch(y)
        y = _rand_pitch(y)
        y = _rand_gain(y)
        y = _mix_noise(y)
        y = np.clip(y, -1.0, 1.0).astype(np.float32)
        sf.write(os.path.join(NEG_DIR, f"unk_mine_{i:03d}_aug{j:02d}.wav"), y, SR_)
        n_aug += 1

total = len(glob.glob(os.path.join(NEG_DIR, "unk_mine_*.wav")))
print(f"[5.8] mined negatives added: {total}（原件 {len(copied)} + 扩增 {n_aug}）")
assert total >= len(copied) * 5, "数量异常，把上方输出发我"

[5.8] 发现 mined 文件 4 条：
     mine_000_打开.wav
     mine_001_关闭.wav
     mine_002_关闭.wav
     mine_003_关闭.wav
[5.8] 负样本目录：/content/my_dataset/_unknown
[5.8] 原件已拷贝 4 条
[5.8] mined negatives added: 44（原件 4 + 扩增 40）


Cell 5.9（新增 v5）：板端真关闭样本接入训练集（1 条，走标准管线）

In [ ]:
import shutil, os
os.makedirs(f"{DATA_DIR}/关闭", exist_ok=True)   # v6: WANTED 已不含"关闭"，目录需手动建
dst = f"{DATA_DIR}/关闭/关闭_板端_000.wav"
shutil.copy("/content/板端关闭_入训1.wav", dst)
print("已接入:", dst)

Cell 5.10（新增 v6）：剪字器——从"关闭"剪出"关"（含板端入训样本），并制作"关"考题

In [ ]:
# ===== v6：从"关闭"剪出"关" =====
def cut_first_syllable(x):
    rms = librosa.feature.rms(y=x, frame_length=256, hop_length=128)[0]
    thr = rms.max() * 0.2
    voiced = rms > thr
    padded = np.concatenate([[False], voiced, [False]])
    starts = np.where(~padded[:-1] & padded[1:])[0]
    ends = np.where(padded[:-1] & ~padded[1:])[0]
    if len(starts) == 0: return None
    s, e = starts[0], ends[0]
    seg = x[s*128 : e*128 + int(0.08*SAMPLE_RATE)]   # 留 80ms 尾巴，保住鼻音韵尾
    if len(seg) < int(0.15*SAMPLE_RATE): return None  # 太短=剪坏了，丢弃
    return to_1s(seg.astype(np.float32))

os.makedirs(f"{DATA_DIR}/关", exist_ok=True)
ok = 0
for f in sorted(glob.glob(f"{DATA_DIR}/关闭/关闭_*.wav")):
    seg = cut_first_syllable(load_wav(f))
    if seg is None:
        print("  剪坏丢弃:", os.path.basename(f)); continue
    out = f.replace("/关闭/", "/关/").replace("关闭_", "关_")
    tf.io.write_file(out, tf.audio.encode_wav(seg[:, None], SAMPLE_RATE))
    ok += 1
print(f"剪出 关 {ok} 条（含板端样本 关_板端_000.wav）")

# 考题：从板端真"关闭"里剪出"关"（同样流程，保证考场公平）
x, _ = librosa.load("/content/板端关闭_考题_勿入训.wav", sr=SAMPLE_RATE)
seg = cut_first_syllable(to_1s(x))
tf.io.write_file("/content/关_考题.wav", tf.audio.encode_wav(seg[:, None], SAMPLE_RATE))
print("关 考题已生成")

Cell 6：留一人测试集（不参与训练）

In [ ]:
test_tags = set(PERSON_TAGS[-TEST_PEOPLE:])
print(f"held-out person: {test_tags}")

X_test_list, y_test_list = [], []
for w in WANTED:
    for tag in test_tags:
        for f in glob.glob(f"{DATA_DIR}/{w}/{w}_{tag}_*.wav"):
            X_test_list.append(extract_log_mel(load_wav(f))); y_test_list.append(w2i[w])
print(f"test clips: {len(X_test_list)}")

held-out person: {'男5'}
test clips: 87


Cell 7：训练/验证集组装（先分文件再增强，无泄漏；silence 用真底噪）

In [ ]:
rng = random.Random(42)
X_tr_l, y_tr_l, X_val_l, y_val_l = [], [], [], []

def augment(a):
    aug = a.copy()
    if random.random() < 0.5:
        r = random.choice([0.85, 0.9, 1.1, 1.15])
        aug = librosa.resample(aug, orig_sr=SAMPLE_RATE, target_sr=int(SAMPLE_RATE*r))
        aug = to_1s(aug)
    s = random.randint(-200, 200)
    if s > 0: aug = np.pad(aug,(s,0))[:len(aug)]
    elif s < 0: aug = np.pad(aug,(0,-s))[-len(aug):]
    aug = mix_with_noise(aug, random.uniform(5, 20))
    aug *= random.uniform(0.4, 1.4)
    aug = librosa.effects.pitch_shift(aug, sr=SAMPLE_RATE, n_steps=random.uniform(-3, 3))
    return to_1s(aug)

for w in WANTED:
    fs = [f for f in glob.glob(f"{DATA_DIR}/{w}/{w}_*.wav")
          if os.path.basename(f).split("_")[1] not in test_tags]
    rng.shuffle(fs)
    fs = [f for f in fs if '_板端_' not in f] + [f for f in fs if '_板端_' in f]  # 板端样本钉死在训练侧
    k = max(1, int(len(fs)*0.15))
    for f in fs[k:]:
        a = load_wav(f)
        X_tr_l.append(extract_log_mel(a)); y_tr_l.append(w2i[w])
        for _ in range(8):
            X_tr_l.append(extract_log_mel(augment(a))); y_tr_l.append(w2i[w])
        for _ in range(2):
            X_tr_l.append(extract_log_mel(boardify(a))); y_tr_l.append(w2i[w])
    for f in fs[:k]:
        X_val_l.append(extract_log_mel(load_wav(f))); y_val_l.append(w2i[w])

unk_fs = glob.glob(f"{UNK_DIR}/*.wav"); rng.shuffle(unk_fs)
k = max(1, int(len(unk_fs)*0.15))
for f in unk_fs[k:]:
    X_tr_l.append(extract_log_mel(load_wav(f))); y_tr_l.append(w2i["negative"])
for f in unk_fs[:k]:
    X_val_l.append(extract_log_mel(load_wav(f))); y_val_l.append(w2i["negative"])

def make_silence():
    n = noise_pool[random.randrange(len(noise_pool))]
    st = random.randint(0, len(n)-SAMPLE_RATE) if len(n) > SAMPLE_RATE else 0
    return to_1s(n[st:st+SAMPLE_RATE]*random.uniform(0.4, 1.2)).astype(np.float32)

for _ in range(400):
    X_tr_l.append(extract_log_mel(make_silence())); y_tr_l.append(w2i["negative"])
for _ in range(60):
    X_val_l.append(extract_log_mel(make_silence())); y_val_l.append(w2i["negative"])

X_tr = np.array(X_tr_l).astype(np.float32); y_tr = np.array(y_tr_l)
X_val = np.array(X_val_l).astype(np.float32); y_val = np.array(y_val_l)

# ===== v5 微调：归一化冻结为封版 v4 常数（板端零改动前提，勿改回重算）=====
_p = np.load("/content/kimi_kws_0724_4cls_v4_norm_params.npz")
mean, std = _p["mean"], _p["std"]
X_tr = (X_tr-mean)/(std+1e-8)
X_val = (X_val-mean)/(std+1e-8)
np.savez("/content/norm_params.npz", mean=mean, std=std)  # 供导出 cell 使用（内容与 v4 相同）

X_test_arr = np.array(X_test_list).astype(np.float32)
X_test_arr = (X_test_arr-mean)/(std+1e-8)
y_test_arr = np.array(y_test_list)

y_tr_oh = tf.keras.utils.to_categorical(y_tr, NUM_CLASSES)
y_val_oh = tf.keras.utils.to_categorical(y_val, NUM_CLASSES)
print(f"train: {X_tr.shape[0]}  val: {X_val.shape[0]}  held-out test: {len(X_test_arr)}")
from collections import Counter
print("train dist:", {i2w[c]: n for c, n in sorted(Counter(y_tr).items())})
print("val dist:  ", {i2w[c]: n for c, n in sorted(Counter(y_val).items())})

train: 8908  val: 785  held-out test: 87
train dist: {'negative': 3976, '打开': 1629, '关闭': 1656, '唤醒': 1647}
val dist:   {'negative': 690, '打开': 31, '关闭': 32, '唤醒': 32}


Cell 8：模型（未改动）

In [ ]:
# ===== v6：从头训练（词义已变，不基于 v4 微调）；结构未改 =====
model = models.Sequential([
    layers.Input(shape=(NUM_FRAMES,NUM_MEL_BINS)),
    layers.Reshape((NUM_FRAMES,NUM_MEL_BINS,1)),
    layers.Conv2D(16,(10,8),strides=(2,2),padding="same",activation="relu"),
    layers.DepthwiseConv2D((3,3),padding="same",activation="relu"),
    layers.Conv2D(32,(1,1),padding="same",activation="relu"),
    layers.AveragePooling2D(pool_size=(2,2)),
    layers.Flatten(),
    layers.Dense(64,activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES,activation="softmax"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ reshape (Reshape)               │ (None, 49, 40, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 25, 20, 16)     │         1,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ depthwise_conv2d                │ (None, 25, 20, 16)     │           160 │
│ (DepthwiseConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 25, 20, 32)     │           544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ average_pooling2d               │ (None, 12, 10, 32)     │             0 │
│ (AveragePooling2D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3840)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │       245,824 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 248,084 (969.08 KB)

 Trainable params: 248,084 (969.08 KB)

 Non-trainable params: 0 (0.00 B)

Cell 9：训练（加 class_weight）

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_tr)
class_weight = {i: float(c) for i, c in enumerate(cw)}
print("class_weight:", {i2w[i]: round(class_weight[i],2) for i in range(NUM_CLASSES)})

cb = [
    tf.keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, monitor="val_loss", mode="min"),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5, monitor="val_loss", mode="min"),
]
history = model.fit(X_tr, y_tr_oh, batch_size=50, epochs=100,
                    validation_data=(X_val, y_val_oh), callbacks=cb,
                    class_weight=class_weight, verbose=1)
loss, acc = model.evaluate(X_val, y_val_oh, verbose=0)
print(f"val acc (leakage-free): {acc*100:.2f}%")
model.save("/content/kws_float_model.h5")

class_weight: {'negative': 0.56, '打开': 1.37, '关闭': 1.34, '唤醒': 1.35}
Epoch 1/100
179/179 ━━━━━━━━━━━━━━━━━━━━ 13s 56ms/step - accuracy: 0.8187 - loss: 0.5353 - val_accuracy: 0.9325 - val_loss: 0.2223 - learning_rate: 0.0010
Epoch 2/100
179/179 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.9365 - loss: 0.1999 - val_accuracy: 0.9911 - val_loss: 0.0395 - learning_rate: 0.0010
Epoch 3/100
179/179 ━━━━━━━━━━━━━━━━━━━━ 11s 60ms/step - accuracy: 0.9553 - loss: 0.1377 - val_accuracy: 0.9898 - val_loss: 0.0349 - learning_rate: 0.0010
Epoch 4/100
179/179 ━━━━━━━━━━━━━━━━━━━━ 20s 57ms/step - accuracy: 0.9646 - loss: 0.1037 - val_accuracy: 0.9898 - val_loss: 0.0386 - learning_rate: 0.0010
Epoch 5/100
179/179 ━━━━━━━━━━━━━━━━━━━━ 10s 54ms/step - accuracy: 0.9728 - loss: 0.0824 - val_accuracy: 0.9873 - val_loss: 0.0379 - learning_rate: 0.0010
Epoch 6/100
179/179 ━━━━━━━━━━━━━━━━━━━━ 11s 59ms/step - accuracy: 0.9759 - loss: 0.0725 - val_accuracy: 0.9898 - val_loss: 0.0308 - learning_rate: 0.0010
E

val acc (leakage-free): 99.49%


Cell 10：INT8 量化

In [ ]:
def rep_dataset():
    for i in range(min(len(X_tr),500)):
        yield [X_tr[i:i+1].astype(np.float32)]
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = rep_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8; converter.inference_output_type = tf.int8
tflite_q = converter.convert()
with open("/content/kws_quant.tflite","wb") as f: f.write(tflite_q)
print(f"INT8: {len(tflite_q)/1024:.1f} KB")

Saved artifact at '/tmp/tmp5sgrvord'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 49, 40), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  137844199908624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199903632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199909008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199909392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199899408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199910352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199900944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199910544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199908816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137844199909968: TensorSpec(shape=(), dtype=tf.resource, name=None)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


INT8: 251.3 KB


In [ ]:
# ========== 桥接 cell v2：重建量化推理环境（磁盘/内存双通道） ==========
# 位置：Cell 10 之后、量化阈值表 cell 之前跑一次
import glob, os
import tensorflow as tf

# --- 通道1：磁盘上找任意 .tflite（不限文件名） ---
cands = sorted(glob.glob("/content/**/*.tflite", recursive=True),
               key=os.path.getmtime)
tfl_path = cands[-1] if cands else None

# --- 通道2：磁盘没有，从内存变量里找字节流落盘 ---
if tfl_path is None:
    print("磁盘上没找到 .tflite，改从内存变量里找……")
    mem = {k: v for k, v in list(globals().items())
           if isinstance(v, (bytes, bytearray)) and len(v) > 50000}
    if mem:
        name, data = max(mem.items(), key=lambda kv: len(kv[1]))
        print(f"找到内存变量 {name}（{len(data)/1024:.1f} KB），写入 /content/v4_int8.tflite")
        tfl_path = "/content/v4_int8.tflite"
        with open(tfl_path, "wb") as f:
            f.write(data)

assert tfl_path is not None, "磁盘和内存都没找到量化模型！把 Cell 10 的完整代码发我，我看它到底存哪了"

print("加载：", tfl_path)
interp = tf.lite.Interpreter(model_path=tfl_path)
interp.allocate_tensors()

inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]
in_s, in_z = inp["quantization"]

print(f"输入 dtype={inp['dtype']}  形状={inp['shape']}")
print(f"in_s={in_s}  in_z={in_z}")
assert in_s is not None and in_s > 0, "输入量化参数异常！把上面打印和 Cell 10 代码一起发我"

加载： /content/kws_quant.tflite
输入 dtype=<class 'numpy.int8'>  形状=[ 1 49 40]
in_s=0.03337131068110466  in_z=40


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
# 补丁：补输出端量化参数
out_s, out_z = out["quantization"]
print(f"out_s={out_s}  out_z={out_z}")

out_s=0.00390625  out_z=-128


cell10.5量化阈值表

In [ ]:
probs_q = []
for i in range(len(X_val)):
    q = np.clip(np.round(X_val[i]/in_s+in_z), -128, 127).astype(np.int8)
    interp.set_tensor(inp["index"], q.reshape(1, NUM_FRAMES, NUM_MEL_BINS))
    interp.invoke()
    raw = interp.get_tensor(out["index"])[0]
    if out["dtype"] == np.int8: raw = (raw.astype(np.float32)-out_z)*out_s
    e = np.exp(raw - raw.max()); probs_q.append(e/e.sum())
probs_q = np.array(probs_q)
kw_mask = y_val >= 1; neg_mask = y_val < 1
print("INT8 quantized sweep")
print(" thr   keyword_detect   false_trigger")
for thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    det = sum(1 for i in range(len(y_val)) if kw_mask[i] and probs_q[i][y_val[i]] >= thr) / kw_mask.sum()
    fa = sum(1 for i in range(len(y_val)) if neg_mask[i] and probs_q[i][1:].max() >= thr) / neg_mask.sum()
    print(f"{thr:.2f}     {det*100:6.1f}%          {fa*100:6.2f}%")

INT8 quantized sweep
 thr   keyword_detect   false_trigger
0.30       95.8%            0.00%
0.40       93.7%            0.00%
0.50        0.0%            0.00%
0.60        0.0%            0.00%
0.70        0.0%            0.00%
0.80        0.0%            0.00%


Cell 11：跨人测试（量化模型）


In [ ]:
interp = tf.lite.Interpreter(model_content=tflite_q)
interp.allocate_tensors()
inp = interp.get_input_details()[0]; out = interp.get_output_details()[0]
in_s,in_z = inp["quantization"]; out_s,out_z = out["quantization"]

def run_one(x):
    q = np.clip(np.round(x/in_s+in_z), -128, 127).astype(np.int8)
    interp.set_tensor(inp["index"], q.reshape(1,NUM_FRAMES,NUM_MEL_BINS)); interp.invoke()
    raw = interp.get_tensor(out["index"])
    if out["dtype"] == np.int8: raw = (raw.astype(np.float32)-out_z)*out_s
    return raw[0]

correct = 0
per_class = {c:[0,0] for c in range(NUM_CLASSES)}
for i in range(len(X_test_arr)):
    pred = np.argmax(run_one(X_test_arr[i]))
    gt = y_test_arr[i]
    per_class[gt][1] += 1
    if pred == gt: correct += 1; per_class[gt][0] += 1

print(f"\n===== held-out person test ({list(test_tags)}) =====")
print(f"overall: {correct}/{len(X_test_arr)} = {correct/len(X_test_arr)*100:.2f}%")
for cid in range(NUM_CLASSES):
    c, t = per_class[cid]
    if t > 0: print(f"  {i2w[cid]}: {c}/{t} = {c/t*100:.1f}%")


===== held-out person test (['男5']) =====
overall: 84/87 = 96.55%
  打开: 30/30 = 100.0%
  关闭: 25/27 = 92.6%
  唤醒: 29/30 = 96.7%


Cell 11.5（新增 v5）：考题——板端真关闭（从未入训），关闭必须第一

In [ ]:
x, _ = librosa.load("/content/关_考题.wav", sr=SAMPLE_RATE)
fx = (extract_log_mel(to_1s(x)) - mean.reshape(40,)) / (std.reshape(40,) + 1e-8)
raw = run_one(fx.astype(np.float32))
e = np.exp(raw - raw.max()); prob = e/e.sum()
for cid in range(NUM_CLASSES):
    print(f"{i2w[cid]}: {prob[cid]:.3f}")
print("判定:", "PASS（关第一）" if prob.argmax() == w2i["关"] else "FAIL")

Cell 12：导出 C 数组 + 下载清单（部署用）

In [ ]:
import binascii, shutil
TAG = f"kimi_kws_{DATE_STR}_4cls_v6guan"

TFLITE_OUT = f"/content/{TAG}_int8.tflite"
NORM_OUT   = f"/content/{TAG}_norm_params.npz"
CC_OUT     = f"/content/{TAG}_model_data.cc"
H5_OUT     = f"/content/{TAG}_float.h5"

shutil.copy("/content/kws_quant.tflite", TFLITE_OUT)
shutil.copy("/content/norm_params.npz", NORM_OUT)
shutil.copy("/content/kws_float_model.h5", H5_OUT)

hexstr = binascii.hexlify(tflite_q).decode()
bytes_list = [f"0x{hexstr[i:i+2]}," for i in range(0, len(hexstr), 2)]
rows = ["  " + "".join(bytes_list[i:i+12]) for i in range(0, len(bytes_list), 12)]
c_code = ("#include <cstdint>\n\nalignas(8) const unsigned char g_kws_model_data[] = {\n"
          + "\n".join(rows) + "\n};\n"
          + f"const unsigned int g_kws_model_data_len = {len(tflite_q)};\n")
with open(CC_OUT, "w") as f: f.write(c_code)

from google.colab import files
for f in [TFLITE_OUT, NORM_OUT, CC_OUT, H5_OUT]:
    files.download(f)
print("downloaded:", TAG, "x4 files")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded: kimi_kws_0724_4cls_v4 x4 files
